# Entraînement LSTM sur les 6 variantes SIEVE

Ce notebook reprend l'organisation du notebook RNN corrigé fourni : même recherche de `processed_data`, mêmes noms de fichiers `*_X_train.csv`, `*_X_val.csv`, `*_X_test.csv`, même lecture CSV robuste pour préserver les virgules présentes dans les logs, même tokenisation et mêmes métriques.

La différence principale est l'architecture : `nn.LSTM` remplace `nn.RNN`.


In [1]:
import os
import re
import json
import random
import csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Appareil utilisé :", DEVICE)

VARIANTES = [
    "output_0", "output_0.1", "output_0.2",
    "output_0.3", "output_0.4", "output_0.5"
]

BATCH_SIZE = 32
EPOCHS = 3
MAX_LENGTH = 128
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
LEARNING_RATE = 1e-3

os.makedirs("resultats_lstm", exist_ok=True)


Appareil utilisé : mps


In [3]:
def tokenizer(texte):
    # Conserver les mots, nombres, adresses IP et signes utiles des logs.
    return re.findall(r"[a-z]+|\d+(?:\.\d+){1,3}|\d+|[^\w\s]", str(texte).lower())


class LogsDataset(Dataset):
    def __init__(self, textes, labels, vocab):
        self.textes = textes
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = tokenizer(self.textes.iloc[idx])
        ids = [self.vocab.get(tok, self.vocab["<UNK>"]) for tok in tokens]
        ids = ids[:MAX_LENGTH]
        longueur = max(1, len(ids))
        ids = ids + [self.vocab["<PAD>"]] * (MAX_LENGTH - len(ids))

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(longueur, dtype=torch.long),
            torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long)
        )


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(HIDDEN_DIM, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)

        # Utiliser la sortie correspondant au dernier vrai token,
        # et non à une position de padding.
        indices = (lengths - 1).to(x.device)
        last_output = output[torch.arange(x.size(0), device=x.device), indices]
        return self.fc(self.dropout(last_output))


def construire_vocab(train_textes):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for texte in train_textes:
        for token in tokenizer(texte):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab


def creer_loader(df, vocab, shuffle=False):
    dataset = LogsDataset(df["log"], df["label"], vocab)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0
    )


def evaluer_modele(model, loader, criterion=None):
    model.eval()
    pertes, vraies, predictions = [], [], []

    with torch.no_grad():
        for x, longueurs, y in loader:
            x = x.to(DEVICE)
            longueurs = longueurs.to(DEVICE)
            y = y.to(DEVICE)

            sorties = model(x, longueurs)

            if criterion is not None:
                pertes.append(criterion(sorties, y).item())

            pred = torch.argmax(sorties, dim=1)
            vraies.extend(y.cpu().numpy())
            predictions.extend(pred.cpu().numpy())

    accuracy = accuracy_score(vraies, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        vraies,
        predictions,
        average="macro",
        zero_division=0
    )
    perte = float(np.mean(pertes)) if pertes else None

    return perte, accuracy, precision, recall, f1, vraies, predictions


## Chargement des données et entraînement des six variantes

In [5]:
DATA_DIR = next(
    (
        dossier
        for dossier in (Path("../processed_data"), Path("processed_data"))
        if (dossier / f"{VARIANTES[0]}_X_train.csv").is_file()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Impossible de trouver le dossier processed_data. "
        "Place ce notebook dans la même organisation de dossiers que le notebook RNN."
    )


def charger_split(path):
    lignes = []

    with path.open("r", encoding="utf-8", newline="") as fichier:
        lecteur = csv.reader(fichier)
        entete = next(lecteur, None)

        if entete != ["log", "label"]:
            raise ValueError(f"En-tête inattendu dans {path} : {entete}")

        for numero, ligne in enumerate(lecteur, start=2):
            if not ligne:
                continue

            if len(ligne) < 2:
                raise ValueError(f"Ligne {numero} invalide dans {path}")

            # Le dernier champ est le label ; les virgules précédentes
            # appartiennent éventuellement au texte du log.
            log = ",".join(ligne[:-1]).strip()

            try:
                label = int(ligne[-1].strip())
            except ValueError as exc:
                raise ValueError(
                    f"Label invalide à la ligne {numero} dans {path}"
                ) from exc

            lignes.append({"log": log, "label": label})

    return pd.DataFrame(lignes, columns=["log", "label"])


resultats = []

for variante in VARIANTES:
    print("\\n" + "=" * 60)
    print("VARIANTE :", variante)
    print("=" * 60)

    train_path = DATA_DIR / f"{variante}_X_train.csv"
    val_path = DATA_DIR / f"{variante}_X_val.csv"
    test_path = DATA_DIR / f"{variante}_X_test.csv"

    for chemin in [train_path, val_path, test_path]:
        if not chemin.is_file():
            raise FileNotFoundError(
                f"Fichier manquant : {chemin}. Vérifie les fichiers de prétraitement."
            )

    train_df = charger_split(train_path)
    val_df = charger_split(val_path)
    test_df = charger_split(test_path)

    vocab = construire_vocab(train_df["log"])
    num_classes = int(train_df["label"].nunique())

    train_loader = creer_loader(train_df, vocab, shuffle=True)
    val_loader = creer_loader(val_df, vocab)
    test_loader = creer_loader(test_df, vocab)

    model = LSTMClassifier(len(vocab), num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    meilleur_f1_val = -1
    meilleur_etat = None

    for epoch in range(EPOCHS):
        model.train()
        pertes_train = []

        for x, longueurs, y in train_loader:
            x = x.to(DEVICE)
            longueurs = longueurs.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            sorties = model(x, longueurs)
            loss = criterion(sorties, y)
            loss.backward()
            optimizer.step()

            pertes_train.append(loss.item())

        val_loss, val_acc, val_precision, val_recall, val_f1, _, _ = (
            evaluer_modele(model, val_loader, criterion)
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train loss: {np.mean(pertes_train):.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val accuracy: {val_acc:.4f} | "
            f"Val macro-F1: {val_f1:.4f}"
        )

        if val_f1 > meilleur_f1_val:
            meilleur_f1_val = val_f1
            meilleur_etat = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    if meilleur_etat is not None:
        model.load_state_dict(meilleur_etat)

    test_loss, test_acc, test_precision, test_recall, test_f1, y_true, y_pred = (
        evaluer_modele(model, test_loader, criterion)
    )

    print(
        f"TEST | Accuracy: {test_acc:.4f} | "
        f"Macro-F1: {test_f1:.4f}"
    )
    print(classification_report(y_true, y_pred, zero_division=0))

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "vocab_size": len(vocab),
            "num_classes": num_classes,
            "max_length": MAX_LENGTH,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dim": HIDDEN_DIM,
        },
        f"resultats_lstm/lstm_{variante}.pth"
    )

    with open(
        f"resultats_lstm/vocab_{variante}.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(vocab, f, ensure_ascii=False)

    resultats.append({
        "variante": variante,
        "accuracy": test_acc,
        "precision_macro": test_precision,
        "recall_macro": test_recall,
        "f1_macro": test_f1,
        "test_loss": test_loss
    })

    # Sauvegarder à chaque variante pour conserver les résultats
    # si l'exécution est interrompue.
    pd.DataFrame(resultats).to_csv(
        "resultats_lstm/comparaison_lstm.csv",
        index=False
    )

print("\\nEntraînement des six variantes terminé.")


\n============================================================
VARIANTE : output_0
Epoch 1/3 | Train loss: 0.6694 | Val loss: 0.1117 | Val accuracy: 0.9596 | Val macro-F1: 0.9594
Epoch 2/3 | Train loss: 0.0849 | Val loss: 0.0841 | Val accuracy: 0.9661 | Val macro-F1: 0.9658
Epoch 3/3 | Train loss: 0.0578 | Val loss: 0.0651 | Val accuracy: 0.9680 | Val macro-F1: 0.9677
TEST | Accuracy: 0.9672 | Macro-F1: 0.9670
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       175
           1       0.97      0.99      0.98       175
           2       1.00      0.90      0.95       175
           3       0.92      0.99      0.95       175
           4       1.00      0.91      0.95       175
           5       1.00      1.00      1.00       175
           6       1.00      0.77      0.87       175
           7       1.00      1.00      1.00       175
           8       0.81      0.97      0.88       175
           9       1.00      1.00      1.00  

## Tableau comparatif final

In [6]:
comparaison = pd.DataFrame(resultats)
display(comparaison)
comparaison.to_csv("resultats_lstm/comparaison_lstm.csv", index=False)


,variante,accuracy,precision_macro,recall_macro,f1_macro,test_loss
0,output_0,0.967238,0.970229,0.967238,0.967028,0.066993
1,output_0.1,0.954286,0.956342,0.954286,0.954229,0.108692
2,output_0.2,0.936000,0.938498,0.936000,0.935647,0.204206
3,output_0.3,0.936000,0.938803,0.936000,0.935712,0.158683
4,output_0.4,0.936381,0.940093,0.936381,0.935962,0.173668
5,output_0.5,0.918857,0.924698,0.918857,0.918153,0.215858
